# 03b — Input-level modality Shapley (v2)

This notebook computes input-level two-player Shapley values for the image and clinical branches. It keeps the frozen v1 split, base outputs, fusion choice, and probability-level Shapley results unchanged.

**OOF rule:** each validation fold is evaluated with branch models refit on that fold's training rows. The models are discarded after the fold; completed OOF fold outputs are atomically checkpointed and resumed on rerun. Missing tabular input is the mean XGBoost probability over 50 seeded, class-stratified training rows. Missing image input is a deterministic Gaussian-blurred raw image evaluated by the same fine-tuned ViT.

**Test rule:** use the saved final branch models and backgrounds sampled only from training rows. Test results are reporting-only. This notebook does not tune or select a model on test data.

The ViT recipe is the two-stage recipe in notebook 01 (head-only stage 1, full-model fine-tuning stage 2). The input-level OOF run therefore retrains five ViTs and may require a GPU and substantial time. Results are saved as `outputs/modality_shapley_v2.npz`; no `_v1` artifact is overwritten.

CUDA is detected through the selected PyTorch build and NVIDIA driver; a separate `CUDA_PATH` is not needed for a prebuilt CUDA-enabled PyTorch wheel. Setup performs a CUDA smoke test and stops rather than silently running the five-fold workload on CPU.

In [1]:
import copy
import gc
import json
import os
import platform
import random
from pathlib import Path

ROOT = Path.cwd()
OUT_DIR = ROOT / 'outputs'
CACHE_DIR = ROOT / 'cache'
TMP_DIR = ROOT / 'tmp'
TMP_DIR.mkdir(parents=True, exist_ok=True)
os.environ['TEMP'] = os.environ['TMP'] = str(TMP_DIR)

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torchvision
from PIL import Image, ImageFilter
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from torchvision import models
from torchvision.transforms import v2
import xgboost
from xgboost import XGBClassifier

from common import FEATURES, IMG_ROOT, META, TARGET, build_image_index

SEED = 42
N_BACKGROUND = 50
BLUR_RADIUS = 32
B = 1000
torch.hub.set_dir(str(ROOT / 'torch_home' / 'hub'))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

required = [
    OUT_DIR / 'base_outputs_v1.npz',
    OUT_DIR / 'fusion_v1.npz',
    OUT_DIR / 'modality_shapley_v1.npz',
    OUT_DIR / 'split_v1.json',
    OUT_DIR / 'xgb_final_v1.joblib',
    OUT_DIR / 'vit_final_v1.pt',
    CACHE_DIR / 'images_uint8_224.pt',
    Path(META),
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Required v1 input(s) missing: ' + ', '.join(missing))

base = np.load(OUT_DIR / 'base_outputs_v1.npz', allow_pickle=True)
CLASSES = np.asarray(base['classes']).astype(str)
C = len(CLASSES)
assert CLASSES.tolist() == ['ACK', 'BCC', 'MEL', 'NEV', 'SCC', 'SEK']
oof_T, oof_I = base['oof_xgb'], base['oof_vit']
te_T, te_I = base['test_xgb'], base['test_vit']
y_tr, y_te, fold = base['y_train'].astype(int), base['y_test'].astype(int), base['fold'].astype(int)
assert (len(y_tr), len(y_te)) == (1837, 461), (len(y_tr), len(y_te))
assert fold.shape == y_tr.shape and np.isin(fold, np.arange(5)).all()
assert oof_T.shape == oof_I.shape == (len(y_tr), C)
assert te_T.shape == te_I.shape == (len(y_te), C)
for name, prob in [('oof_xgb', oof_T), ('oof_vit', oof_I), ('test_xgb', te_T), ('test_vit', te_I)]:
    assert np.isfinite(prob).all() and np.allclose(prob.sum(axis=1), 1.0, atol=1e-5), name

with open(OUT_DIR / 'split_v1.json', encoding='utf-8') as f:
    split = json.load(f)
train_ids = split['train_img_ids']
test_ids = split['test_img_ids']
assert len(train_ids) == len(y_tr) and len(test_ids) == len(y_te)
data = pd.read_csv(META)
data[TARGET] = data[TARGET].replace({'BOD': 'SCC'})
assert data['img_id'].is_unique
data = data.set_index('img_id', drop=False)
train_df = data.loc[train_ids].reset_index(drop=True)
test_df = data.loc[test_ids].reset_index(drop=True)
assert train_df['img_id'].tolist() == train_ids
assert test_df['img_id'].tolist() == test_ids
le = LabelEncoder().fit(train_df[TARGET])
assert np.array_equal(le.classes_.astype(str), CLASSES)
y_tr_check = le.transform(train_df[TARGET])
y_te_check = le.transform(test_df[TARGET])
assert np.array_equal(y_tr_check, y_tr) and np.array_equal(y_te_check, y_te)
expected_fold = np.asarray([split['fold_by_img_id'][img_id] for img_id in train_ids])
assert np.array_equal(expected_fold, fold)
pat_tr = train_df['patient_id'].astype(str).to_numpy()
pat_te = test_df['patient_id'].astype(str).to_numpy()
X_tr = pd.get_dummies(train_df[FEATURES], dummy_na=True).astype(float)
X_te = pd.get_dummies(test_df[FEATURES], dummy_na=True).reindex(columns=X_tr.columns, fill_value=0).astype(float)
assert X_tr.shape[0] == len(y_tr) and X_te.shape[0] == len(y_te)

image_cache_path = CACHE_DIR / 'images_uint8_224.pt'
image_cache = torch.load(image_cache_path, map_location='cpu', weights_only=True)
IMG_TR, IMG_TE = image_cache['tr'], image_cache['te']
assert tuple(IMG_TR.shape) == (len(y_tr), 3, 224, 224)
assert tuple(IMG_TE.shape) == (len(y_te), 3, 224, 224)
assert IMG_TR.dtype == IMG_TE.dtype == torch.uint8
img_index = build_image_index(IMG_ROOT)
assert all(img_id in img_index for img_id in train_ids + test_ids), 'image index does not cover frozen split'
for sample_indices, frame, images, label in [
    ([0, len(train_ids) // 2, len(train_ids) - 1], train_df, IMG_TR, 'train'),
    ([0, len(test_ids) // 2, len(test_ids) - 1], test_df, IMG_TE, 'test'),
]:
    for idx in sample_indices:
        cached = images[idx].permute(1, 2, 0).numpy()
        with Image.open(img_index[frame.iloc[idx]['img_id']]) as im:
            expected = np.asarray(im.convert('RGB').resize((224, 224)))
        assert np.array_equal(cached, expected), f'{label} cache order mismatch at row {idx}'

fusion = np.load(OUT_DIR / 'fusion_v1.npz', allow_pickle=True)
selected = str(np.asarray(fusion['selected']).item())
assert selected == 'CWF', selected
wT_full = np.asarray(fusion['w_xgb'], dtype=float)
assert wT_full.shape == (C,) and np.all((wT_full >= 0) & (wT_full <= 1))
shapley_v1 = np.load(OUT_DIR / 'modality_shapley_v1.npz', allow_pickle=True)
assert np.array_equal(shapley_v1['classes'].astype(str), CLASSES)
prior = np.bincount(y_tr, minlength=C).astype(float) / len(y_tr)
uniform = np.full(C, 1.0 / C)
assert np.allclose(shapley_v1['prior'], prior)

if torch.version.cuda is None or not torch.cuda.is_available():
    raise RuntimeError('CUDA-enabled PyTorch and a visible NVIDIA GPU are required for the five-fold ViT run.')
device = torch.device('cuda')
try:
    torch.empty(1, device=device).add_(1)
    torch.cuda.synchronize()
except RuntimeError as exc:
    raise RuntimeError(f'CUDA smoke test failed on {torch.cuda.get_device_name(0)}: {exc}') from exc
torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))
print(CLASSES.tolist(), '| train', len(y_tr), '| test', len(y_te), '| device', device)
print('CUDA device:', torch.cuda.get_device_name(0), '| PyTorch CUDA runtime:', torch.version.cuda)
print('cache row order spot checks passed; CWF weights:', np.round(wT_full, 3).tolist())
print('versions:', {
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__,
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'xgboost': xgboost.__version__,
})

['ACK', 'BCC', 'MEL', 'NEV', 'SCC', 'SEK'] | train 1837 | test 461 | device cuda
CUDA device: NVIDIA GeForce RTX 5050 Laptop GPU | PyTorch CUDA runtime: 12.8
cache row order spot checks passed; CWF weights: [0.515, 0.526, 0.415, 0.498, 0.452, 0.373]
versions: {'python': '3.11.15', 'numpy': '2.4.3', 'pandas': '3.0.1', 'scikit-learn': '1.7.1', 'torch': '2.12.0.dev20260401+cu128', 'torchvision': '0.27.0.dev20260401+cu128', 'xgboost': '3.2.0'}


In [2]:
def macro_f1(y_true, prediction):
    prediction = np.asarray(prediction)
    if prediction.ndim == 2:
        prediction = prediction.argmax(axis=1)
    return float(f1_score(y_true, prediction, average='macro'))


def cwf_weights(PT, PI, y):
    recall_T = recall_score(y, PT.argmax(axis=1), average=None, labels=np.arange(C), zero_division=0)
    recall_I = recall_score(y, PI.argmax(axis=1), average=None, labels=np.arange(C), zero_division=0)
    return np.where(recall_T + recall_I > 0, recall_T / np.maximum(recall_T + recall_I, 1e-12), 0.5)


def cwf_apply(PT, PI, wT):
    mixed = np.asarray(wT)[None, :] * PT + (1.0 - np.asarray(wT)[None, :]) * PI
    return mixed / np.maximum(mixed.sum(axis=1, keepdims=True), 1e-12)


def shapley_two(PT, PI, wT, bT, bI):
    PT, PI = np.asarray(PT, dtype=float), np.asarray(PI, dtype=float)
    if PT.ndim != 2 or PI.shape != PT.shape:
        raise ValueError('PT and PI must have matching (n_rows, n_classes) shapes')
    BT = np.broadcast_to(np.asarray(bT, dtype=float), PT.shape)
    BI = np.broadcast_to(np.asarray(bI, dtype=float), PI.shape)
    coalitions = {
        'TI': cwf_apply(PT, PI, wT),
        'bI': cwf_apply(BT, PI, wT),
        'Tb': cwf_apply(PT, BI, wT),
        'bb': cwf_apply(BT, BI, wT),
    }
    prediction = coalitions['TI'].argmax(axis=1)
    rows = np.arange(len(prediction))
    payoff = {key: value[rows, prediction] for key, value in coalitions.items()}
    phiT = 0.5 * (payoff['TI'] - payoff['bI']) + 0.5 * (payoff['Tb'] - payoff['bb'])
    phiI = 0.5 * (payoff['TI'] - payoff['Tb']) + 0.5 * (payoff['bI'] - payoff['bb'])
    assert np.allclose(phiT + phiI, payoff['TI'] - payoff['bb'], atol=1e-10)
    denominator = np.abs(phiT) + np.abs(phiI)
    rho = np.divide(np.abs(phiT), denominator, out=np.zeros_like(phiT), where=denominator > 1e-12)
    result = {
        'pred': prediction,
        'phiT': phiT,
        'phiI': phiI,
        'rho': rho,
        'f': payoff['TI'],
        'base': payoff['bb'],
    }
    assert all(np.isfinite(value).all() for value in result.values())
    return result


def input_level_shapley(PT, PI, wT, background_T, blurred_I):
    return shapley_two(PT, PI, wT, background_T, blurred_I)


def sample_background_indices(y, pool_indices, n=N_BACKGROUND, seed=SEED):
    pool_indices = np.asarray(pool_indices, dtype=int)
    if n <= 0 or len(pool_indices) < n:
        raise ValueError(f'background needs at least {n} rows; pool has {len(pool_indices)}')
    if len(np.unique(pool_indices)) != len(pool_indices):
        raise ValueError('background pool indices must be unique')
    rng = np.random.default_rng(seed)
    present_classes = np.unique(np.asarray(y)[pool_indices])
    if len(present_classes) == 0:
        raise ValueError('background pool contains no labels')
    quotas = np.full(len(present_classes), n // len(present_classes), dtype=int)
    quotas[:n % len(present_classes)] += 1
    selected = []
    for cls, quota in zip(present_classes, quotas):
        candidates = pool_indices[np.asarray(y)[pool_indices] == cls]
        take = min(int(quota), len(candidates))
        if take:
            selected.extend(rng.choice(candidates, size=take, replace=False).tolist())
    selected = np.asarray(selected, dtype=int)
    if len(selected) < n:
        remaining = np.setdiff1d(pool_indices, selected, assume_unique=False)
        selected = np.concatenate([selected, rng.choice(remaining, size=n - len(selected), replace=False)])
    result = np.sort(selected.astype(int))
    if len(result) != n or len(np.unique(result)) != n or not np.isin(result, pool_indices).all():
        raise AssertionError('invalid background sample')
    return result

In [3]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
AUG = v2.Compose([
    v2.RandomHorizontalFlip(),
    v2.RandomVerticalFlip(),
    v2.RandomAffine(degrees=25, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
])


def prep(x_uint8, train=False):
    x = x_uint8.to(device, non_blocking=True).float() / 255.0
    if train:
        x = torch.stack([AUG(image) for image in x])
    return (x - MEAN) / STD


def blur_uint8(images):
    blurred = []
    for image in images:
        array = image.cpu().permute(1, 2, 0).numpy()
        pil_image = Image.fromarray(array).filter(ImageFilter.GaussianBlur(radius=BLUR_RADIUS))
        blurred.append(torch.from_numpy(np.asarray(pil_image, dtype=np.uint8).copy()).permute(2, 0, 1))
    return torch.stack(blurred)


def make_vit(pretrained=True):
    weights = models.ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.vit_b_16(weights=weights)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.heads.head = nn.Linear(model.heads.head.in_features, C)
    return model.to(device)


@torch.no_grad()
def predict_vit(model, images_uint8, blur=False, batch_size=32):
    model.eval()
    outputs = []
    for start in range(0, len(images_uint8), batch_size):
        batch = images_uint8[start:start + batch_size]
        if blur:
            batch = blur_uint8(batch)
        outputs.append(torch.softmax(model(prep(batch, train=False)), dim=1).cpu().numpy())
    return np.concatenate(outputs, axis=0)


def train_vit(X_uint8, y, groups, seed=SEED, batch_size=32, verbose=True):
    torch.manual_seed(seed)
    np.random.seed(seed)
    indices = np.arange(len(y))
    tr_i, va_i = next(
        GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=seed).split(indices, y, groups)
    )
    tr_i, va_i = torch.tensor(tr_i), torch.tensor(va_i)
    y_t = torch.tensor(y, dtype=torch.long)
    model = make_vit()
    class_weights = torch.tensor(
        compute_class_weight('balanced', classes=np.arange(C), y=y[tr_i.numpy()]),
        dtype=torch.float,
        device=device,
    )
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    @torch.no_grad()
    def val_loss():
        model.eval()
        total = 0.0
        for start in range(0, len(va_i), 64):
            idx = va_i[start:start + 64]
            loss = criterion(model(prep(X_uint8[idx], train=False)), y_t[idx].to(device))
            total += loss.item() * len(idx)
        return total / len(va_i)

    def run(parameters, learning_rate, max_epochs, patience, tag):
        optimizer = torch.optim.AdamW(parameters, lr=learning_rate, weight_decay=0.05)
        best, best_state, bad = np.inf, copy.deepcopy(model.state_dict()), 0
        for epoch in range(max_epochs):
            model.train()
            permutation = tr_i[torch.randperm(len(tr_i))]
            for start in range(0, len(permutation), batch_size):
                idx = permutation[start:start + batch_size]
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(prep(X_uint8[idx], train=True)), y_t[idx].to(device))
                loss.backward()
                optimizer.step()
            current = val_loss()
            improved = current < best
            if verbose and (epoch % 5 == 0 or improved):
                print(f'    [{tag}] epoch {epoch:3d}  inner-val loss {current:.4f}' + ('  *' if improved else ''))
            if improved:
                best, best_state, bad = current, copy.deepcopy(model.state_dict()), 0
            else:
                bad += 1
                if bad >= patience:
                    break
        model.load_state_dict(best_state)
        return best

    stage1 = run(model.heads.parameters(), 1e-4, 50, 20, 'stage1 head')
    for parameter in model.parameters():
        parameter.requires_grad = True
    stage2 = run(model.parameters(), 1e-6, 100, 20, 'stage2 full')
    val_f1 = macro_f1(y[va_i.numpy()], predict_vit(model, X_uint8[va_i]).argmax(axis=1))
    if verbose:
        print(f'  best inner-val loss: stage1 {stage1:.4f} -> stage2 {stage2:.4f} | inner-val macro-F1 {val_f1:.3f}')
    return model


def make_xgb():
    return XGBClassifier(
        objective='multi:softprob',
        num_class=C,
        eval_metric='mlogloss',
        random_state=SEED,
    )


def patient_cluster_bootstrap_mean(values, mask, patients, n_boot=B, seed=SEED):
    values, mask, patients = np.asarray(values), np.asarray(mask, dtype=bool), np.asarray(patients)
    unique_patients = np.unique(patients)
    groups = [np.flatnonzero(patients == patient) for patient in unique_patients]
    rng = np.random.default_rng(seed)
    estimates = []
    for _ in range(n_boot):
        sampled = rng.integers(0, len(groups), size=len(groups))
        idx = np.concatenate([groups[position] for position in sampled])
        selected = idx[mask[idx]]
        if len(selected):
            estimates.append(values[selected].mean())
    if not estimates:
        return np.nan, np.nan
    return tuple(np.percentile(estimates, [2.5, 97.5]))


def patient_cluster_bootstrap_difference(values, correct, patients, n_boot=B, seed=SEED):
    values, correct, patients = np.asarray(values), np.asarray(correct, dtype=bool), np.asarray(patients)
    unique_patients = np.unique(patients)
    groups = [np.flatnonzero(patients == patient) for patient in unique_patients]
    rng = np.random.default_rng(seed)
    estimates = []
    for _ in range(n_boot):
        sampled = rng.integers(0, len(groups), size=len(groups))
        idx = np.concatenate([groups[position] for position in sampled])
        ok = idx[correct[idx]]
        wrong = idx[~correct[idx]]
        if len(ok) and len(wrong):
            estimates.append(values[wrong].mean() - values[ok].mean())
    if not estimates:
        return np.nan, np.nan
    return tuple(np.percentile(estimates, [2.5, 97.5]))


def class_share_table(shares, labels, patients, title, seed):
    rows = []
    for cls, name in enumerate(CLASSES):
        mask = labels == cls
        lower, upper = patient_cluster_bootstrap_mean(shares, mask, patients, seed=seed + cls)
        rows.append({
            'class': name,
            'n': int(mask.sum()),
            'mean_rhoT': float(np.mean(shares[mask])) if mask.any() else np.nan,
            'patient_cluster_CI': (round(lower, 3), round(upper, 3)) if np.isfinite(lower + upper) else ('n/a', 'n/a'),
        })
    print(title)
    print(pd.DataFrame(rows).set_index('class').to_string())


def error_detection_auc(S, y, PT, PI, label):
    error = (S['pred'] != y).astype(int)
    disagreement = (PT.argmax(axis=1) != PI.argmax(axis=1)).astype(float)
    scores = {
        'tabular_share_high': S['rho'],
        'image_contribution_low': -S['phiI'],
        'branch_disagreement': disagreement,
        'fused_confidence_low': -S['f'],
    }
    print(f'--- {label} error-detection AUROC (higher score means more likely wrong; n={len(y)}, errors={error.sum()})')
    for name, score in scores.items():
        print(f'  {name}: {roc_auc_score(error, score):.3f}')
    return disagreement


def fit_error_detectors(X_train, error_train):
    X_train = np.asarray(X_train, dtype=float)
    error_train = np.asarray(error_train, dtype=int)
    if X_train.ndim != 2 or X_train.shape[1] != 4 or len(X_train) != len(error_train):
        raise ValueError('error detector expects four features and one error label per training row')
    if not np.isfinite(X_train).all() or set(np.unique(error_train)) != {0, 1}:
        raise ValueError('error detector training data must be finite and contain both error classes')
    baseline = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight='balanced'),
    )
    all_features = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight='balanced'),
    )
    baseline.fit(X_train[:, [3]], error_train)
    all_features.fit(X_train, error_train)
    return baseline, all_features


def score_error_detectors(models_pair, X_eval, error_eval):
    baseline, all_features = models_pair
    X_eval = np.asarray(X_eval, dtype=float)
    error_eval = np.asarray(error_eval, dtype=int)
    if X_eval.ndim != 2 or X_eval.shape[1] != 4 or len(X_eval) != len(error_eval):
        raise ValueError('error detector evaluation expects four features and one error label per row')
    if not np.isfinite(X_eval).all() or set(np.unique(error_eval)) != {0, 1}:
        raise ValueError('error detector evaluation data must be finite and contain both error classes')
    return {
        'confidence_only_auc': roc_auc_score(
            error_eval, baseline.predict_proba(X_eval[:, [3]])[:, 1]
        ),
        'all_features_auc': roc_auc_score(
            error_eval, all_features.predict_proba(X_eval)[:, 1]
        ),
    }

In [4]:
def run_helper_self_checks():
    assert macro_f1(np.array([0, 1]), np.array([[0.9, 0.1], [0.2, 0.8]])) == 1.0
    pt = np.array([[0.7, 0.3], [0.2, 0.8]])
    pi = np.array([[0.4, 0.6], [0.6, 0.4]])
    wt = np.array([0.6, 0.4])
    result = shapley_two(pt, pi, wt, np.array([0.5, 0.5]), np.array([0.5, 0.5]))
    assert np.allclose(result['phiT'] + result['phiI'], result['f'] - result['base'])
    per_case = input_level_shapley(
        pt, pi, wt, np.array([0.5, 0.5]), np.array([[0.3, 0.7], [0.8, 0.2]])
    )
    assert per_case['rho'].shape == (2,)
    pt_same = np.repeat(pt[:1], len(pt), axis=0)
    zero = shapley_two(pt_same, pt_same, wt, pt_same[0], pt_same[0])
    assert np.isfinite(zero['rho']).all() and np.all(zero['rho'] == 0)
    labels = np.repeat(np.arange(3), 10)
    pool = np.arange(len(labels))
    bg1 = sample_background_indices(labels, pool, n=12, seed=7)
    bg2 = sample_background_indices(labels, pool, n=12, seed=7)
    assert np.array_equal(bg1, bg2)
    assert len(bg1) == 12 and np.isin(bg1, pool).all()
    counts = np.bincount(labels[bg1], minlength=3)
    assert counts.max() - counts.min() <= 1
    train_error = np.tile([0, 1], 6)
    train_features = np.column_stack([
        np.linspace(0.1, 0.9, len(train_error)),
        np.linspace(0.9, 0.1, len(train_error)),
        np.tile([0.0, 1.0], 6),
        np.tile([0.85, 0.25], 6),
    ])
    baseline_model, full_model = fit_error_detectors(train_features, train_error)
    assert baseline_model.named_steps['standardscaler'].n_samples_seen_ == len(train_error)
    assert full_model.named_steps['standardscaler'].n_samples_seen_ == len(train_error)
    test_scores = score_error_detectors(
        (baseline_model, full_model),
        train_features[:4],
        train_error[:4],
    )
    assert set(test_scores) == {'confidence_only_auc', 'all_features_auc'}
    assert np.isfinite(list(test_scores.values())).all()
    print('Shapley and background self-checks passed')


run_helper_self_checks()

Shapley and background self-checks passed


## Probability-level secondary analysis

The v1 probabilities are retained as the secondary comparison. This cell recomputes their prior-baseline Shapley values and checks that the signed values and predictions match the saved v1 artifact before proceeding.

In [5]:
def run_oof_probability_shapley(baseline):
    out = {
        'phiT': np.zeros(len(y_tr)),
        'phiI': np.zeros(len(y_tr)),
        'rho': np.zeros(len(y_tr)),
        'pred': np.zeros(len(y_tr), dtype=int),
        'f': np.zeros(len(y_tr)),
        'base': np.zeros(len(y_tr)),
    }
    for k in range(5):
        train_idx, valid_idx = np.flatnonzero(fold != k), np.flatnonzero(fold == k)
        weight = cwf_weights(oof_T[train_idx], oof_I[train_idx], y_tr[train_idx])
        result = shapley_two(oof_T[valid_idx], oof_I[valid_idx], weight, baseline, baseline)
        for key in out:
            out[key][valid_idx] = result[key]
    return out


prob_oof = run_oof_probability_shapley(prior)
prob_test = shapley_two(te_T, te_I, wT_full, prior, prior)
prob_oof_uniform = run_oof_probability_shapley(uniform)
prob_test_uniform = shapley_two(te_T, te_I, wT_full, uniform, uniform)
assert np.allclose(cwf_weights(oof_T, oof_I, y_tr), wT_full, atol=1e-12)
assert np.allclose(shapley_v1['w_xgb'], wT_full, atol=1e-12)
assert np.allclose(prob_oof['phiT'], shapley_v1['oof_phiT'], atol=1e-8)
assert np.allclose(prob_oof['phiI'], shapley_v1['oof_phiI'], atol=1e-8)
assert np.array_equal(prob_oof['pred'], shapley_v1['oof_pred'])
assert np.allclose(prob_test['phiT'], shapley_v1['test_phiT'], atol=1e-8)
assert np.allclose(prob_test['phiI'], shapley_v1['test_phiI'], atol=1e-8)
assert np.array_equal(prob_test['pred'], shapley_v1['test_pred'])
print('Probability-level signed attributions and predictions match v1.')
for label, prior_result, uniform_result in [
    ('OOF', prob_oof, prob_oof_uniform),
    ('TEST', prob_test, prob_test_uniform),
]:
    print(f'{label} probability baseline robustness: '
          f'Spearman rho={spearmanr(prior_result["rho"], uniform_result["rho"]).statistic:.3f} | '
          f'tabular-phi sign agreement={(np.sign(prior_result["phiT"]) == np.sign(uniform_result["phiT"])).mean():.3f}')

Probability-level signed attributions and predictions match v1.
OOF probability baseline robustness: Spearman rho=0.843 | tabular-phi sign agreement=0.964
TEST probability baseline robustness: Spearman rho=0.833 | tabular-phi sign agreement=0.939


## Cross-fitted input-level analysis

For each held-out fold, train on the other four folds, select the fold's 50 background rows only from that training partition, evaluate complete and blurred images using that fold's ViT, and compute the four coalition values. Completed OOF results are checkpointed after each fold and resumed on rerun.

In [6]:
def empty_result(n_rows):
    return {
        'phiT': np.zeros(n_rows),
        'phiI': np.zeros(n_rows),
        'rho': np.zeros(n_rows),
        'pred': np.zeros(n_rows, dtype=int),
        'f': np.zeros(n_rows),
        'base': np.zeros(n_rows),
        'baseline_T': np.zeros((n_rows, C)),
        'baseline_I': np.zeros((n_rows, C)),
    }


def save_oof_checkpoint(path, fold_assignment, completed_folds, input_oof, fold_w_xgb,
                        oof_background_indices, oof_full_T, oof_full_I):
    payload = {
        'fold_assignment': np.asarray(fold_assignment),
        'completed_folds': np.asarray(completed_folds, dtype=bool),
        'fold_w_xgb': fold_w_xgb,
        'oof_background_indices': oof_background_indices,
        'oof_full_T': oof_full_T,
        'oof_full_I': oof_full_I,
    }
    payload.update({f'input_oof__{key}': value for key, value in input_oof.items()})
    temp_path = path.with_name(path.name + '.tmp')
    try:
        with temp_path.open('wb') as checkpoint_file:
            np.savez_compressed(checkpoint_file, **payload)
        os.replace(temp_path, path)
    finally:
        if temp_path.exists():
            temp_path.unlink()


def restore_oof_checkpoint(path, fold_assignment, input_oof, fold_w_xgb,
                           oof_background_indices, oof_full_T, oof_full_I):
    completed_folds = np.zeros(5, dtype=bool)
    if not path.is_file():
        return completed_folds

    arrays = {
        'fold_w_xgb': fold_w_xgb,
        'oof_background_indices': oof_background_indices,
        'oof_full_T': oof_full_T,
        'oof_full_I': oof_full_I,
    }
    arrays.update({f'input_oof__{key}': value for key, value in input_oof.items()})
    with np.load(path, allow_pickle=False) as checkpoint:
        required = {'fold_assignment', 'completed_folds', *arrays}
        missing = required.difference(checkpoint.files)
        if missing:
            raise ValueError(f'Invalid OOF checkpoint; missing arrays: {sorted(missing)}')
        if not np.array_equal(checkpoint['fold_assignment'], fold_assignment):
            raise ValueError('OOF checkpoint fold assignment does not match this run')
        saved_completed = checkpoint['completed_folds']
        if saved_completed.shape != completed_folds.shape:
            raise ValueError('OOF checkpoint completed-fold shape does not match this run')
        for key, target in arrays.items():
            if checkpoint[key].shape != target.shape:
                raise ValueError(f'OOF checkpoint shape mismatch for {key}')
        for key, target in arrays.items():
            target[...] = checkpoint[key]
        completed_folds[...] = saved_completed.astype(bool)
    return completed_folds


input_oof = empty_result(len(y_tr))
fold_w_xgb = np.zeros((5, C), dtype=float)
oof_background_indices = np.zeros((5, N_BACKGROUND), dtype=int)
oof_full_T = np.zeros_like(oof_T)
oof_full_I = np.zeros_like(oof_I)
CHECKPOINT_PATH = OUT_DIR / 'modality_shapley_v2_oof_checkpoint.npz'
completed_folds = restore_oof_checkpoint(
    CHECKPOINT_PATH, fold, input_oof, fold_w_xgb,
    oof_background_indices, oof_full_T, oof_full_I,
)
if completed_folds.any():
    print(f'Resuming OOF folds; already complete: {np.flatnonzero(completed_folds).tolist()}')

for k in range(5):
    if completed_folds[k]:
        print(f'===== OOF fold {k} already complete; skipping =====')
        continue
    print(f'===== OOF fold {k} =====')
    train_idx, valid_idx = np.flatnonzero(fold != k), np.flatnonzero(fold == k)
    background_idx = sample_background_indices(y_tr, train_idx, n=N_BACKGROUND, seed=SEED + k)
    assert np.isin(background_idx, train_idx).all()
    assert not np.isin(background_idx, valid_idx).any()
    oof_background_indices[k] = background_idx

    tabular_model = make_xgb().fit(X_tr.iloc[train_idx], y_tr[train_idx])
    assert np.array_equal(np.asarray(tabular_model.classes_), np.arange(C))
    full_T = tabular_model.predict_proba(X_tr.iloc[valid_idx])
    background_T = tabular_model.predict_proba(X_tr.iloc[background_idx]).mean(axis=0)

    image_model = train_vit(IMG_TR[train_idx], y_tr[train_idx], pat_tr[train_idx], seed=SEED)
    full_I = predict_vit(image_model, IMG_TR[valid_idx])
    blurred_I = predict_vit(image_model, IMG_TR[valid_idx], blur=True)
    weight = cwf_weights(oof_T[train_idx], oof_I[train_idx], y_tr[train_idx])
    fold_w_xgb[k] = weight
    result = input_level_shapley(full_T, full_I, weight, background_T, blurred_I)
    for key in ('phiT', 'phiI', 'rho', 'pred', 'f', 'base'):
        input_oof[key][valid_idx] = result[key]
    input_oof['baseline_T'][valid_idx] = np.broadcast_to(background_T, (len(valid_idx), C))
    input_oof['baseline_I'][valid_idx] = blurred_I
    oof_full_T[valid_idx], oof_full_I[valid_idx] = full_T, full_I

    delta_T = float(np.max(np.abs(full_T - oof_T[valid_idx])))
    delta_I = float(np.max(np.abs(full_I - oof_I[valid_idx])))
    print(f'  OOF full-input probability max |delta| vs frozen outputs: XGB={delta_T:.6f}, ViT={delta_I:.6f}')
    print(f'  fold CWF tabular weights: {np.round(weight, 3).tolist()}')

    completed_folds[k] = True
    save_oof_checkpoint(
        CHECKPOINT_PATH, fold, completed_folds, input_oof, fold_w_xgb,
        oof_background_indices, oof_full_T, oof_full_I,
    )
    print(f'  Saved completed OOF folds to {CHECKPOINT_PATH}')

    del image_model, tabular_model
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

assert np.isfinite(input_oof['rho']).all()
assert np.allclose(input_oof['phiT'] + input_oof['phiI'], input_oof['f'] - input_oof['base'], atol=1e-10)

Resuming OOF folds; already complete: [0, 1, 2, 3, 4]
===== OOF fold 0 already complete; skipping =====
===== OOF fold 1 already complete; skipping =====
===== OOF fold 2 already complete; skipping =====
===== OOF fold 3 already complete; skipping =====
===== OOF fold 4 already complete; skipping =====


## Test input-level analysis

The final branch checkpoints are loaded once. The B=50 background is sampled from training rows only. The test data is never fitted, sampled, or used to select a model or baseline.

In [10]:
final_xgb_artifact = joblib.load(OUT_DIR / 'xgb_final_v1.joblib')
final_xgb = final_xgb_artifact['model']
assert np.array_equal(np.asarray(final_xgb.classes_), np.arange(C))

test_full_T = final_xgb.predict_proba(X_te)
test_background_indices = sample_background_indices(
    y_tr, np.arange(len(y_tr)), n=N_BACKGROUND, seed=SEED + 10000
)
test_baseline_T = final_xgb.predict_proba(
    X_tr.iloc[test_background_indices]
).mean(axis=0)

final_vit = models.vit_b_16(weights=None)
final_vit.heads.head = nn.Linear(final_vit.heads.head.in_features, C)
final_state = torch.load(
    OUT_DIR / 'vit_final_v1.pt',
    map_location='cpu',
    weights_only=True,
)
final_vit.load_state_dict(final_state)
final_vit = final_vit.to(device)

torch.backends.cudnn.benchmark = True
test_full_I = predict_vit(final_vit, IMG_TE, batch_size=64)
test_blurred_I = predict_vit(
    final_vit, IMG_TE, blur=True, batch_size=64
)

assert test_full_T.shape == te_T.shape and test_full_I.shape == te_I.shape
print(
    'Final checkpoint probability max |delta| vs frozen test outputs:',
    f'XGB={np.max(np.abs(test_full_T - te_T)):.6f},',
    f'ViT={np.max(np.abs(test_full_I - te_I)):.6f}',
)
print(
    'ViT argmax disagreements vs frozen test outputs:',
    f'{np.count_nonzero(test_full_I.argmax(axis=1) != te_I.argmax(axis=1))}/{len(y_te)}',
)

assert np.allclose(
    test_full_T, te_T, atol=1e-6
), 'final XGBoost checkpoint does not match frozen test outputs'

# Allows the observed small probability deviation while still rejecting larger mismatches.
assert np.allclose(
    test_full_I, te_I, atol=1e-3, rtol=0.0
), 'final ViT checkpoint/preprocessing does not match frozen test outputs'

input_test = input_level_shapley(
    test_full_T, test_full_I, wT_full, test_baseline_T, test_blurred_I
)
input_test['baseline_T'] = np.broadcast_to(
    test_baseline_T, (len(y_te), C)
).copy()
input_test['baseline_I'] = test_blurred_I

assert np.isfinite(input_test['rho']).all()
assert np.allclose(
    input_test['phiT'] + input_test['phiI'],
    input_test['f'] - input_test['base'],
    atol=1e-10,
)


Final checkpoint probability max |delta| vs frozen test outputs: XGB=0.000000, ViT=0.000776
ViT argmax disagreements vs frozen test outputs: 1/461


## Class-aware summaries, clustered uncertainty, and error-detection preview

Cluster bootstrap intervals resample patients, not lesions. The test rows are summarized once and do not change any analysis choices. MEL test has only seven examples and is flagged.

Error-detector logistic models are fit on OOF training explanations and evaluated once on the patient-independent test set. We do not cross-validate the logistic model over precomputed OOF explanations: their upstream branch-model fits can include patients in a different CV validation split.

In [11]:
class_share_table(input_oof['rho'], y_tr, pat_tr, 'OOF input-level share by true class', SEED + 200)
class_share_table(input_oof['rho'], input_oof['pred'], pat_tr, 'OOF input-level share by predicted class', SEED + 300)
class_share_table(input_test['rho'], y_te, pat_te, 'TEST input-level share by true class', SEED + 400)
class_share_table(input_test['rho'], input_test['pred'], pat_te, 'TEST input-level share by predicted class', SEED + 500)
print('CWF tabular weight by class (full OOF for test; mean cross-fitted fold weight for OOF):')
print(pd.DataFrame({
    'class': CLASSES,
    'w_xgb_test': wT_full,
    'mean_w_xgb_oof_folds': fold_w_xgb.mean(axis=0),
}).set_index('class').round(3).to_string())
print(f'MEL test count: {(y_te == np.where(CLASSES == "MEL")[0][0]).sum()} (small; interpret cautiously)')

for label, result, y, patients in [
    ('OOF', input_oof, y_tr, pat_tr),
    ('TEST', input_test, y_te, pat_te),
]:
    correct = result['pred'] == y
    lower, upper = patient_cluster_bootstrap_difference(result['rho'], correct, patients)
    print(f'{label}: mean rho_T wrong - correct = '
          f'{result["rho"][~correct].mean() - result["rho"][correct].mean():.3f}; '
          f'patient-cluster 95% bootstrap CI [{lower:.3f}, {upper:.3f}]')
    error_detection_auc(result, y, oof_full_T if label == 'OOF' else test_full_T,
                        oof_full_I if label == 'OOF' else test_full_I, label)

disagreement_oof = (oof_full_T.argmax(axis=1) != oof_full_I.argmax(axis=1)).astype(float)
disagreement_test = (test_full_T.argmax(axis=1) != test_full_I.argmax(axis=1)).astype(float)
error_oof = (input_oof['pred'] != y_tr).astype(int)
error_test = (input_test['pred'] != y_te).astype(int)
X_error_oof = np.column_stack([input_oof['rho'], input_oof['phiI'], disagreement_oof, input_oof['f']])
X_error_test = np.column_stack([input_test['rho'], input_test['phiI'], disagreement_test, input_test['f']])
error_detectors = fit_error_detectors(X_error_oof, error_oof)
test_error_auc = score_error_detectors(error_detectors, X_error_test, error_test)
print('TEST error AUROC (one held-out evaluation; train fit on OOF features): '
      'confidence-only logistic=%.3f | all features=%.3f'
      % (test_error_auc['confidence_only_auc'], test_error_auc['all_features_auc']))


def compare_probability_shares(input_result, probability_result, labels, title):
    rho_in = input_result['rho']
    rho_prob = probability_result['rho']
    phi_in = input_result['phiT']
    phi_prob = probability_result['phiT']
    print(f'{title} input-vs-probability share agreement: '
          f'Spearman={spearmanr(rho_in, rho_prob).statistic:.3f} | '
          f'tabular-phi sign agreement={(np.sign(phi_in) == np.sign(phi_prob)).mean():.3f}')
    for cls, name in enumerate(CLASSES):
        mask = labels == cls
        if mask.sum() < 2:
            corr = np.nan
        else:
            corr = spearmanr(rho_in[mask], rho_prob[mask]).statistic
        sign = (np.sign(phi_in[mask]) == np.sign(phi_prob[mask])).mean() if mask.any() else np.nan
        print(f'  {name}: n={mask.sum()} | Spearman={corr:.3f} | sign agreement={sign:.3f}')


compare_probability_shares(input_oof, prob_oof, y_tr, 'OOF by true class')
compare_probability_shares(input_test, prob_test, y_te, 'TEST by true class')

np.savez(
    OUT_DIR / 'modality_shapley_v2.npz',
    classes=CLASSES,
    prior=prior,
    uniform=uniform,
    w_xgb=wT_full,
    oof_fold_w_xgb=fold_w_xgb,
    oof_fold=fold,
    oof_y=y_tr,
    test_y=y_te,
    oof_background_indices=oof_background_indices,
    test_background_indices=test_background_indices,
    oof_input_phiT=input_oof['phiT'],
    oof_input_phiI=input_oof['phiI'],
    oof_input_rho=input_oof['rho'],
    oof_input_pred=input_oof['pred'],
    oof_input_f=input_oof['f'],
    oof_input_base=input_oof['base'],
    oof_input_baseline_T=input_oof['baseline_T'],
    oof_input_baseline_I=input_oof['baseline_I'],
    test_input_phiT=input_test['phiT'],
    test_input_phiI=input_test['phiI'],
    test_input_rho=input_test['rho'],
    test_input_pred=input_test['pred'],
    test_input_f=input_test['f'],
    test_input_base=input_test['base'],
    test_input_baseline_T=input_test['baseline_T'],
    test_input_baseline_I=input_test['baseline_I'],
    oof_prob_phiT=prob_oof['phiT'],
    oof_prob_phiI=prob_oof['phiI'],
    oof_prob_rho=prob_oof['rho'],
    oof_prob_pred=prob_oof['pred'],
    oof_prob_f=prob_oof['f'],
    oof_prob_base=prob_oof['base'],
    test_prob_phiT=prob_test['phiT'],
    test_prob_phiI=prob_test['phiI'],
    test_prob_rho=prob_test['rho'],
    test_prob_pred=prob_test['pred'],
    test_prob_f=prob_test['f'],
    test_prob_base=prob_test['base'],
    oof_prob_uniform_rho=prob_oof_uniform['rho'],
    test_prob_uniform_rho=prob_test_uniform['rho'],
    library_versions=np.asarray(json.dumps({
        'python': platform.python_version(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit-learn': sklearn.__version__,
        'torch': torch.__version__,
        'torchvision': torchvision.__version__,
        'xgboost': xgboost.__version__,
    })),
)
saved = np.load(OUT_DIR / 'modality_shapley_v2.npz', allow_pickle=False)
for key in ('oof_input_phiT', 'oof_input_phiI', 'oof_input_rho', 'oof_input_pred',
            'test_input_phiT', 'test_input_phiI', 'test_input_rho', 'test_input_pred'):
    assert len(saved[key]) == len(y_tr) if key.startswith('oof_') else len(saved[key]) == len(y_te), key
    assert np.isfinite(saved[key]).all(), key
print('saved outputs/modality_shapley_v2.npz')

OOF input-level share by true class
         n  mean_rhoT patient_cluster_CI
class                                   
ACK    603   0.611199     (0.594, 0.629)
BCC    671   0.588512     (0.569, 0.608)
MEL     45   0.394988     (0.291, 0.504)
NEV    195   0.699708      (0.66, 0.738)
SCC    143   0.615269      (0.57, 0.663)
SEK    180   0.513857     (0.465, 0.566)
OOF input-level share by predicted class
         n  mean_rhoT patient_cluster_CI
class                                   
ACK    636   0.634678     (0.617, 0.652)
BCC    692   0.584747     (0.566, 0.605)
MEL     39   0.505161      (0.36, 0.646)
NEV    202   0.703303     (0.668, 0.737)
SCC    105   0.594276     (0.534, 0.651)
SEK    163   0.402892      (0.354, 0.45)
TEST input-level share by true class
         n  mean_rhoT patient_cluster_CI
class                                   
ACK    127   0.586754     (0.541, 0.628)
BCC    174   0.612816     (0.579, 0.646)
MEL      7   0.277938     (0.109, 0.552)
NEV     49   0.750436    